# Café model training and benchmark workbench
Train **YOLO11s, YOLO26s and exact RT-DETRv2-R18** on one audited dataset, compare validation results, then evaluate the frozen candidates once on test data. Export candidate weights for later integration.

Designed for the RTX 3050 Laptop (6 GB): start at 512 px, batch 1–2, workers 0. It also runs on Kaggle/Colab GPUs. Public photos are bootstrap data; they cannot establish overhead café accuracy, tracking quality, preparation counts or sales.

**Start:** use a separate Python environment, install the dependencies below, restart the kernel, edit Settings, then Run All. Heavy actions are off by default. This notebook is self-contained: upload this one file to Kaggle/Colab. Keep downloaded data, checkpoints and reports outside Git.

Dataset research and source links: [dataset guide](../docs/research/training_datasets.md). The default Kaggle Cup run uses **only `["cup"]`**. HF COCO also supports person/cup/bottle when you explicitly change the source and target catalog. Never combine incompletely annotated classes.

Stages: settings → acquire/import → audit/prepare → pretrained baseline → train → validation → freeze/test → export. Change settings by starting a **new run**; never reuse test results to tune thresholds.

In [ ]:
from pathlib import Path
import collections, csv, hashlib, importlib.metadata, json, math, random, statistics
import shutil, time, uuid, zipfile
from datetime import datetime, timezone

# Settings: edit here, then restart/run all.
INSTALL_PACKAGES = False
RUN_DOWNLOADS = False
RUN_PRETRAINED = False
RUN_TRAINING = False
RUN_FINAL_TEST = False
RUN_EXPORT = False
RUN_VIDEO = False

SOURCE = "kaggle_cup"           # kaggle_cup | hf_coco | coco_json | yolo
TARGET_CLASSES = ["cup"]
MODEL_NAMES = ["yolo11s", "yolo26s", "rtdetr_v2_r18"]
DATASET_ROOT = None              # existing/pre-attached dataset, otherwise download explicitly
YOLO_CLASS_NAMES = {31: "cup"}  # Kaggle Cup source ID; use a list/dict for another dataset
YOLO_FULLY_ANNOTATED = ["cup"]   # declare only classes actually exhaustively labeled
ALLOW_EMPTY_LABELS = False      # opt in only after checking empty TXT = verified negative
GROUPS_CSV = None                # image,group columns; REQUIRED for own video frames
REAL_CAFE_DATA = False
HF_TRAIN_LIMIT = 1000
HF_TEST_LIMIT = 200
LOCAL_LIMIT_PER_SPLIT = 3000
SEED = 42
IMAGE_SIZE = 512
EPOCHS = 20
YOLO_BATCH = 2
RT_BATCH = 1
GRAD_ACCUMULATION = 4
PATIENCE = 5
LEARNING_RATE = 0.0001
AMP = True                      # false for CPU; deployment latency precision recorded separately
SCORE_THRESHOLD = 0.25
INFERENCE_SCORE_FLOOR = 0.001
DEVICE = "cuda:0"                # "cpu" works, but training will be much slower
WARMUP_IMAGES = 10
LATENCY_REPEATS = 3
VIDEO_PATH = None
VIDEO_MAX_FRAMES = 300
VIDEO_MODEL = "yolo11s"

HF_DATASET = "detection-datasets/coco"
HF_DATASET_REVISION = "cf0b22332314a937e9dc8a1957b21725430bb41d"
RT_REPO = "PekingU/rtdetr_v2_r18vd"
RT_REVISION = "5650961749fa93567c0d46fc7f43ea4f9e914107"
KAGGLE_CUP = "samuelayman/cup-dataset/versions/1"
# Optional local checkpoint overrides (trusted publisher files only).
PRETRAINED_PATHS = {}           # {"yolo11s": "/path/yolo11s.pt", ...}
RT_LOCAL_PATH = None

base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else (
    Path("/content") if Path("/content").is_dir() else Path.cwd())
# Locate the project from VS Code/Jupyter launched at the repository or notebooks directory.
project = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "coffee_ops").is_dir()), None)
WORK_ROOT = (project / "var/notebook-experiments") if project else base / "coffee_ops_experiments"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_ROOT / RUN_ID
PREPARED = RUN_DIR / "prepared"
RECORDS = []
ARTIFACTS = {}
VALIDATION_RESULTS = {}
TEST_RESULTS = {}
print("New run:", RUN_DIR, "| all heavy actions require their settings flags.")

## Isolated environment
Locally use `py -3.13 -m venv .venv-notebook`, then install the matched Torch/TorchVision pair and `requirements-notebook.txt` as documented in the README. Select that interpreter as the notebook kernel. Do not install over the application's .venv.

On Kaggle/Colab, keep their working GPU Torch/TorchVision pair if the native NMS check passes; install only missing optional packages. The optional cell below changes the **current kernel environment**, and requires a kernel restart afterwards. It does not install/replace Torch. Notebook training APIs are pinned; installed package versions are saved with results.

Downloads and training can be substantial. HF streaming bounds the number of retained images, **not network bytes**: Parquet row groups and caches may be larger. Kaggle COCO is ~27.6 GB; attach it as Input instead of downloading it by default.

In [ ]:
if INSTALL_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "ultralytics==8.4.204", "transformers==4.57.6", "num2words==0.5.14",
        "datasets==4.8.5", "torchmetrics==1.9.0", "faster-coco-eval==1.8.0",
        "kagglehub==1.0.3", "ijson==3.6.0"])
    print("Restart the kernel before continuing.")

In [ ]:
def environment():
    import torch, torchvision
    # A metadata-only version check misses mismatched native binaries.
    torchvision.ops.nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5)
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable: select a GPU kernel or explicitly set DEVICE='cpu'.")
    versions = {}
    for package in ["torch", "torchvision", "ultralytics", "transformers", "datasets",
                    "torchmetrics", "faster-coco-eval", "Pillow", "numpy", "kagglehub"]:
        try: versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: versions[package] = "missing"
    return {"versions": versions, "device": DEVICE,
            "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.startswith("cuda") else None,
            "image_size": IMAGE_SIZE, "amp": AMP and DEVICE.startswith("cuda")}

if RUN_PRETRAINED or RUN_TRAINING or RUN_FINAL_TEST:
    print(json.dumps(environment(), indent=2))

## Dataset acquisition and canonical labels
Supported imports: pinned HF COCO streaming (mirror **XYXY**), original COCO instance JSON (**XYWH**, category names mapped explicitly), and YOLO TXT (normalized center/width/height). COCO crowd images are skipped when using original annotations. The HF mirror omits crowd flags: its metrics are a **mirror/subset benchmark**, not official COCO leaderboard evaluation.

All target classes must be fully annotated in every imported image. Missing TXT files are errors; empty TXT requires explicit verification. For personal videos provide `GROUPS_CSV` mapping every relative image path to a recording/session group. Split recordings, not adjacent frames. Near-duplicate visual similarity is **not** checked automatically.

Default class ordering is an experiment choice. Export metadata uses the actual trained ordering; application configuration must match it later.
**Observed download limitation:** the mirror row API currently rejects ~485 MB row groups (300 MB scan limit). Pinned Parquet streaming is supported, but the local first-row transfer stalled during verification. Use a pre-attached Kaggle COCO dataset or Kaggle Cup for a faster first run; streaming sample limits do not bound download bytes.


In [ ]:
COCO_NAMES = ("person,bicycle,car,motorcycle,airplane,bus,train,truck,boat,traffic light,"
"fire hydrant,stop sign,parking meter,bench,bird,cat,dog,horse,sheep,cow,elephant,bear,zebra,"
"giraffe,backpack,umbrella,handbag,tie,suitcase,frisbee,skis,snowboard,sports ball,kite,"
"baseball bat,baseball glove,skateboard,surfboard,tennis racket,bottle,wine glass,cup,fork,"
"knife,spoon,bowl,banana,apple,sandwich,orange,broccoli,carrot,hot dog,pizza,donut,cake,chair,"
"couch,potted plant,bed,dining table,toilet,tv,laptop,mouse,remote,keyboard,cell phone,"
"microwave,oven,toaster,sink,refrigerator,book,clock,vase,scissors,teddy bear,hair drier,toothbrush").split(",")
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch",
           "pottedplant": "potted plant", "diningtable": "dining table", "tvmonitor": "tv"}

def canonical_name(name):
    name = str(name).strip().lower()
    return ALIASES.get(name, name)

def class_mapping(source_names, targets):
    if not targets or len(set(targets)) != len(targets):
        raise ValueError("Target classes must be nonempty and unique.")
    names = {int(i): canonical_name(n) for i, n in (
        source_names.items() if isinstance(source_names, dict) else enumerate(source_names))}
    if any(list(names.values()).count(t) != 1 for t in targets):
        raise ValueError("Each target must occur exactly once in the source class catalog.")
    return {i: targets.index(n) for i, n in names.items() if n in targets}

def xyxy_box(box, width, height, form="xyxy"):
    if len(box) != 4 or not all(math.isfinite(float(x)) for x in box):
        raise ValueError("Box must contain four finite numbers.")
    a, b, c, d = map(float, box)
    if width <= 0 or height <= 0:
        raise ValueError("Invalid image dimensions.")
    if form == "xywh": a, b, c, d = a, b, a + c, b + d
    elif form == "yolo":
        if not all(0 <= x <= 1 for x in [a, b, c, d]):
            raise ValueError("YOLO coordinates must be normalized.")
        a, b, c, d = (a-c/2)*width, (b-d/2)*height, (a+c/2)*width, (b+d/2)*height
    elif form != "xyxy": raise ValueError("Unknown box format.")
    # Tiny floating-point excursions only; do not silently repair bad annotations.
    eps = 1e-3
    if a < -eps or b < -eps or c > width+eps or d > height+eps or c <= a or d <= b:
        raise ValueError("Out-of-bounds or degenerate box.")
    return [max(0., a), max(0., b), min(float(width), c), min(float(height), d)]

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

def pixel_hash(path):
    from PIL import Image
    with Image.open(path) as im:
        im = im.convert("RGB")
        return hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()

def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")

def make_record(path, boxes, labels, annotated, source, split=None, group=None):
    from PIL import Image
    path = Path(path).resolve()
    with Image.open(path) as im: width, height = im.size
    if len(boxes) != len(labels): raise ValueError("Box/label count mismatch.")
    if not set(TARGET_CLASSES) <= set(annotated):
        raise ValueError("Incomplete class coverage: cup-only labels cannot train person/bottle.")
    checked = [xyxy_box(b, width, height) for b in boxes]
    if any(isinstance(x, bool) or not isinstance(x, int) or x < 0 or x >= len(TARGET_CLASSES) for x in labels):
        raise ValueError("Invalid target class ID.")
    digest = pixel_hash(path)
    return {"image": str(path), "width": width, "height": height, "boxes": checked,
            "labels": labels, "fully_annotated_for": list(annotated), "source": source,
            "split": split, "group": group or digest, "sha256_pixels": digest}

def load_groups(path):
    if path is None: return {}
    with open(path, newline="", encoding="utf-8-sig") as f:
        pairs = [(r["image"].replace("\\", "/"), r["group"]) for r in csv.DictReader(f)]
    if len(dict(pairs)) != len(pairs) or any(not a or not b for a, b in pairs):
        raise ValueError("Group CSV has duplicate/missing entries.")
    return dict(pairs)

def import_yolo(root, source_names, annotated, groups=None, allow_empty=False, limit=3000):
    root, groups = Path(root).resolve(), groups or {}
    mapping = class_mapping(source_names, TARGET_CLASSES)
    if not set(TARGET_CLASSES) <= set(annotated):
        raise ValueError("Incomplete class annotation coverage.")
    images = sorted(p for p in root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if not images: raise ValueError("No image files found.")
    records, counts = [], collections.Counter()
    for path in images:
        rel = path.relative_to(root)
        parts = list(rel.parts)
        split = next(({"valid": "val", "validation": "val"}.get(p, p)
                      for p in parts[:-1] if p in {"train", "val", "valid", "validation", "test"}), None)
        if counts[split] >= limit: continue
        label_parts = ["labels" if p == "images" else p for p in parts]
        candidates = {root.joinpath(*label_parts).with_suffix(".txt"),
                      path.parent / "labels" / (path.stem+".txt")}
        existing = [p for p in candidates if p.is_file()]
        if not existing: raise ValueError(f"Missing label for: {path}")
        if len(existing) != 1: raise ValueError(f"Ambiguous labels for: {path}")
        label = existing[0]
        lines = label.read_text(encoding="utf-8-sig").strip().splitlines()
        if not lines and not allow_empty: raise ValueError(f"Unverified negative: {label}")
        from PIL import Image
        with Image.open(path) as im: width, height = im.size
        boxes, labels = [], []
        for line in lines:
            values = line.split()
            if len(values) != 5: raise ValueError(f"Expected YOLO detection rows: {label}")
            sid_float = float(values[0]); sid = int(sid_float)
            if sid != sid_float or sid not in (source_names if isinstance(source_names,dict) else range(len(source_names))):
                raise ValueError("Unknown source label ID; inspect dataset before mapping.")
            box = xyxy_box(list(map(float, values[1:])), width, height, "yolo")
            if sid in mapping: boxes.append(box); labels.append(mapping[sid])
        key = rel.as_posix()
        if REAL_CAFE_DATA and key not in groups: raise ValueError(f"Missing session group: {key}")
        records.append(make_record(path, boxes, labels, annotated, "yolo:" + str(root),
                                   split, groups.get(key)))
        counts[split] += 1
    return records

def import_coco_json(root, limit=3000, groups=None):
    # Stream 470-MB train annotations: never load the full document into RAM.
    import ijson
    root, groups = Path(root), groups or {}
    annotations = sorted(root.rglob("instances_*.json"))
    if not annotations: raise ValueError("No instances_*.json found.")
    records = []
    for ann_path in annotations:
        part = "test" if "val" in ann_path.stem else "train"
        if not any(s in ann_path.stem for s in ["train", "val"]): continue
        with ann_path.open("rb") as f: cats = list(ijson.items(f, "categories.item"))
        mapping = class_mapping({int(c["id"]): c["name"] for c in cats}, TARGET_CLASSES)
        with ann_path.open("rb") as f:
            images = {}
            for im in ijson.items(f, "images.item"):
                images[int(im["id"])] = im
                if len(images) >= limit: break
        labels = {i: [] for i in images}; crowd = set()
        with ann_path.open("rb") as f:
            for ann in ijson.items(f, "annotations.item"):
                iid, cid = int(ann["image_id"]), int(ann["category_id"])
                if iid not in images or cid not in mapping: continue
                if ann.get("iscrowd", 0): crowd.add(iid); continue
                im = images[iid]
                labels[iid].append((xyxy_box(ann["bbox"], int(im["width"]), int(im["height"]), "xywh"), mapping[cid]))
        folder = "val2017" if part == "test" else "train2017"
        candidates = list(root.rglob(folder))
        dirs = [p for p in candidates if p.is_dir()]
        if len(dirs) != 1: raise ValueError(f"Expected one {folder} directory.")
        for iid, im in images.items():
            if iid in crowd: continue
            path = dirs[0] / im["file_name"]
            key = path.relative_to(root).as_posix()
            if REAL_CAFE_DATA and key not in groups: raise ValueError(f"Missing session group: {key}")
            rows = labels[iid]
            records.append(make_record(path, [x[0] for x in rows], [x[1] for x in rows],
                TARGET_CLASSES, "coco-json:" + str(ann_path), part, groups.get(key)))
    return records

def acquire_hf(train_limit, test_limit, output):
    from datasets import load_dataset
    from PIL import Image
    mapping = class_mapping(COCO_NAMES, TARGET_CLASSES)
    output = Path(output); output.mkdir(parents=True, exist_ok=False)
    records = []
    for source_split, assigned, limit in [("train", "train", train_limit), ("val", "test", test_limit)]:
        if limit < 1: raise ValueError("Dataset limits must be positive.")
        stream = load_dataset(HF_DATASET, revision=HF_DATASET_REVISION, split=source_split, streaming=True)
        for n, row in enumerate(stream):
            if n >= limit: break
            im = row["image"].convert("RGB")
            path = output / f"{source_split}-{row['image_id']}.png"
            im.save(path)
            obj = row["objects"]; boxes, labels = [], []
            # This pinned mirror uses absolute XYXY, unlike official COCO JSON.
            if len(obj["bbox"]) != len(obj["category"]): raise ValueError("Malformed HF annotations.")
            for box, category in zip(obj["bbox"], obj["category"]):
                cid = int(category)
                if not 0 <= cid < len(COCO_NAMES): raise ValueError("Unknown HF category.")
                checked = xyxy_box(box, im.width, im.height)
                if cid in mapping: boxes.append(checked); labels.append(mapping[cid])
            records.append(make_record(path, boxes, labels, TARGET_CLASSES,
                f"hf:{HF_DATASET}@{HF_DATASET_REVISION}", assigned))
    return records

def split_and_audit(records, seed=42):
    if not records: raise ValueError("No records to audit.")
    records = [dict(r) for r in records]
    seen, groups = {}, collections.defaultdict(list)
    for r in records:
        if not set(TARGET_CLASSES) <= set(r["fully_annotated_for"]):
            raise ValueError("Incomplete annotation coverage.")
        if r["sha256_pixels"] != pixel_hash(r["image"]): raise ValueError("Image changed after import.")
        # Identical content is rejected even under different filenames/encodings.
        if r["sha256_pixels"] in seen: raise ValueError("Duplicate image pixels; resolve labels/splits explicitly.")
        seen[r["sha256_pixels"]] = r
        if r["split"] not in {None, "train", "val", "test"}: raise ValueError("Unknown split.")
        groups[r["group"]].append(r)
    for group in groups.values():
        known = {r["split"] for r in group if r["split"] is not None}
        if len(known) > 1: raise ValueError("Recording group crosses dataset splits.")
        if known:
            for r in group: r["split"] = next(iter(known))
    # Respect fixed val/test. Derive missing partitions only from train/unassigned groups.
    have = {r["split"] for r in records}
    pool = [k for k, rows in groups.items() if rows[0]["split"] in {None, "train"}]
    random.Random(seed).shuffle(pool)
    missing = [s for s in ["val", "test"] if s not in have]
    if len(pool) < len(missing) + 1: raise ValueError("Too few independent training groups for disjoint splits.")
    for split in missing:
        count = max(1, round(len(pool) * (0.15 if split == "val" else 0.15)))
        for key in pool[:count]:
            for r in groups[key]: r["split"] = split
        pool = pool[count:]
    for r in records:
        if r["split"] is None: r["split"] = "train"
    counts = {s: collections.Counter(c for r in records if r["split"] == s for c in r["labels"])
              for s in ["train", "val", "test"]}
    if any(not any(r["split"] == s for r in records) for s in counts): raise ValueError("Empty split.")
    if any(counts["train"][i] == 0 for i in range(len(TARGET_CLASSES))):
        raise ValueError("At least one target class has no training examples.")
    warnings = [f"{s}: {name} has only {counts[s][i]} boxes"
                for s in counts for i, name in enumerate(TARGET_CLASSES) if counts[s][i] < 20]
    return records, {"images": dict(collections.Counter(r["split"] for r in records)),
                     "boxes": {s: {name: counts[s][i] for i, name in enumerate(TARGET_CLASSES)} for s in counts},
                     "warnings": warnings, "real_cafe_data": REAL_CAFE_DATA,
                     "negative_images": {s:sum(not r["labels"] for r in records if r["split"]==s) for s in counts},
                     "small_boxes_under_32px": {s:sum((b[2]-b[0])*(b[3]-b[1])<32**2
                         for r in records if r["split"]==s for b in r["boxes"]) for s in counts},
                     "near_duplicates_checked": False, "promotion_eligible": False}

def dataset_fingerprint(records):
    portable = [{k:r[k] for k in ["sha256_pixels","width","height","boxes","labels","split","group","source"]}
                for r in records]
    return hashlib.sha256(json.dumps({"classes":TARGET_CLASSES,"records":portable}, sort_keys=True).encode()).hexdigest()

def prepare_dataset(records, destination):
    from PIL import Image
    import yaml
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    prepared = []
    for r in records:
        split = r["split"]; key = r["sha256_pixels"]
        image_dir, label_dir = destination / "images" / split, destination / "labels" / split
        image_dir.mkdir(parents=True, exist_ok=True); label_dir.mkdir(parents=True, exist_ok=True)
        path = image_dir / (key + ".png")
        # Work on copies, preserving raw pixel geometry and removing orientation metadata.
        with Image.open(r["image"]) as im: im.convert("RGB").save(path)
        text = []
        for b, c in zip(r["boxes"], r["labels"]):
            x1,y1,x2,y2=b; w,h=r["width"],r["height"]
            text.append(f"{c} {(x1+x2)/2/w:.9f} {(y1+y2)/2/h:.9f} {(x2-x1)/w:.9f} {(y2-y1)/h:.9f}")
        (label_dir / (key+".txt")).write_text("\n".join(text), encoding="utf-8")
        prepared.append(dict(r, original_image=r["image"], image=str(path.resolve()),
                             label_file=str((label_dir/(key+".txt")).resolve()),
                             label_sha256=file_hash(label_dir/(key+".txt"))))
    config = {"path": str(destination.resolve()), "train": "images/train", "val": "images/val",
              "test": "images/test", "names": dict(enumerate(TARGET_CLASSES))}
    (destination / "dataset.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    # Hash a portable manifest (exclude machine-specific paths).
    portable = [{k: r[k] for k in ["sha256_pixels","width","height","boxes","labels","split","group","source"]}
                for r in prepared]
    dataset_hash = dataset_fingerprint(prepared)
    json_write(destination / "manifest.json", {"dataset_sha256": dataset_hash,
               "classes": TARGET_CLASSES, "records": prepared})
    return prepared, dataset_hash


In [ ]:
def checkpoint_hash(path):
    path = Path(path)
    if path.is_file(): return file_hash(path)
    if not path.is_dir(): raise FileNotFoundError(path)
    files = sorted(p for p in path.rglob("*") if p.is_file())
    if not files: raise ValueError("Empty checkpoint directory.")
    h = hashlib.sha256()
    for p in files:
        h.update(p.relative_to(path).as_posix().encode()); h.update(file_hash(p).encode())
    return h.hexdigest()

def pretrained_location(name):
    if name == "rtdetr_v2_r18":
        if RT_LOCAL_PATH: return str(RT_LOCAL_PATH)
        from huggingface_hub import snapshot_download
        cache = str(project / "var/model-benchmark/huggingface/hub") if project else None
        return snapshot_download(RT_REPO, revision=RT_REVISION, cache_dir=cache,
                                 local_files_only=not RUN_DOWNLOADS)
    if name not in {"yolo11s", "yolo26s"}: raise ValueError("Unsupported model.")
    candidates = [Path(PRETRAINED_PATHS[name])] if name in PRETRAINED_PATHS else []
    if project: candidates.append(project / "var/model-benchmark/weights" / (name+".pt"))
    for p in candidates:
        if p.is_file(): return str(p.resolve())
    if not RUN_DOWNLOADS: raise FileNotFoundError(f"Set PRETRAINED_PATHS or enable downloads for {name}.")
    from ultralytics.utils.downloads import safe_download
    dest = WORK_ROOT / "publisher-checkpoints"; dest.mkdir(parents=True, exist_ok=True)
    safe_download(url=f"https://github.com/ultralytics/assets/releases/download/v8.4.0/{name}.pt",
                  dir=dest, file=name+".pt")
    return str((dest / (name+".pt")).resolve())

def load_adapter(name, checkpoint, trained=False):
    import torch
    if name not in MODEL_NAMES: raise ValueError("Model not enabled.")
    if name.startswith("yolo"):
        from ultralytics import YOLO
        model = YOLO(str(checkpoint))
        names = model.names
        mapping = class_mapping(names, TARGET_CLASSES)
        if trained and [canonical_name(names[i]) for i in range(len(names))] != TARGET_CLASSES:
            raise ValueError("Checkpoint class order does not match trained experiment.")
        def predict(image):
            result = model.predict(image, imgsz=IMAGE_SIZE, device=DEVICE, quantize=16 if AMP and DEVICE.startswith("cuda") else 32,
                                   rect=False, conf=INFERENCE_SCORE_FLOOR, iou=0.7, max_det=300, verbose=False)[0]
            rows = result.boxes
            return tidy_predictions(rows.xyxy.cpu(), rows.conf.cpu(), rows.cls.int().cpu(), mapping, image.size)
    else:
        from transformers import AutoImageProcessor, AutoModelForObjectDetection
        processor = AutoImageProcessor.from_pretrained(checkpoint, local_files_only=True, use_fast=False)
        processor.size = {"height": IMAGE_SIZE, "width": IMAGE_SIZE}
        model = AutoModelForObjectDetection.from_pretrained(checkpoint, local_files_only=True).to(DEVICE).eval()
        mapping = class_mapping(model.config.id2label, TARGET_CLASSES)
        if trained and [canonical_name(model.config.id2label[i]) for i in range(len(TARGET_CLASSES))] != TARGET_CLASSES:
            raise ValueError("RT-DETR class order mismatch.")
        half = AMP and DEVICE.startswith("cuda")
        if half: model.half()
        def predict(image):
            batch = processor(images=image, return_tensors="pt").to(DEVICE)
            if half: batch["pixel_values"] = batch["pixel_values"].half()
            with torch.inference_mode():
                outputs = model(**batch)
            result = processor.post_process_object_detection(outputs, threshold=INFERENCE_SCORE_FLOOR,
                target_sizes=torch.tensor([[image.height, image.width]], device=DEVICE))[0]
            return tidy_predictions(result["boxes"].cpu(), result["scores"].cpu(),
                                    result["labels"].cpu(), mapping, image.size)
    return {"predict": predict, "model": model, "checkpoint": str(checkpoint), "name": name}

def tidy_predictions(boxes, scores, labels, mapping, size):
    import torch
    out_b, out_s, out_l = [], [], []
    width, height = size
    for box, score, label in zip(boxes.tolist(), scores.tolist(), labels.tolist()):
        if int(label) not in mapping: continue
        if not all(math.isfinite(v) for v in [*box, score]): raise ValueError("Nonfinite prediction.")
        x1,y1,x2,y2 = box
        clipped = [max(0.,min(width,x1)),max(0.,min(height,y1)),
                   max(0.,min(width,x2)),max(0.,min(height,y2))]
        if clipped[2] <= clipped[0] or clipped[3] <= clipped[1]: continue
        out_b.append(clipped); out_s.append(score); out_l.append(mapping[int(label)])
    return {"boxes": torch.tensor(out_b, dtype=torch.float32).reshape(-1,4),
            "scores": torch.tensor(out_s, dtype=torch.float32), "labels": torch.tensor(out_l, dtype=torch.int64)}

def matching_counts(pred, truth, threshold):
    import torch
    from scipy.optimize import linear_sum_assignment
    from torchvision.ops import box_iou
    result = {}
    for cid in range(len(TARGET_CLASSES)):
        pb = pred["boxes"][(pred["labels"] == cid) & (pred["scores"] >= threshold)]
        tb = truth["boxes"][truth["labels"] == cid]
        matched = 0
        if len(pb) and len(tb):
            iou = box_iou(pb, tb).numpy()
            # Maximize eligible match count first, IoU second.
            weight = (iou >= .5) * (max(iou.shape) + 1 + iou)
            a,b = linear_sum_assignment(weight, maximize=True)
            matched = sum(iou[x,y] >= .5 for x,y in zip(a,b))
        result[cid] = [int(matched), len(pb)-int(matched), len(tb)-int(matched)]
    return result

def public_numbers(value):
    import torch
    if isinstance(value, torch.Tensor): return public_numbers(value.tolist())
    if isinstance(value, dict): return {k: public_numbers(v) for k,v in value.items()}
    if isinstance(value, list): return [public_numbers(v) for v in value]
    if isinstance(value, float) and (not math.isfinite(value) or value == -1): return None
    return value

def assert_dataset_unchanged(records):
    for r in records:
        if pixel_hash(r["image"]) != r["sha256_pixels"]: raise ValueError("Prepared image changed.")
        if "label_file" in r and file_hash(r["label_file"]) != r["label_sha256"]:
            raise ValueError("Prepared label changed.")

def benchmark_adapter(adapter, records, split, dataset_hash):
    import torch
    from PIL import Image
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    if dataset_fingerprint(records) != dataset_hash: raise ValueError("Dataset annotations changed.")
    rows = [r for r in records if r["split"] == split]
    if not rows: raise ValueError("Evaluation split is empty.")
    assert_dataset_unchanged(rows)
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True,
                                  backend="faster_coco_eval")
    totals = {i:[0,0,0] for i in range(len(TARGET_CLASSES))}
    samples = []
    def sync():
        if DEVICE.startswith("cuda"): torch.cuda.synchronize(DEVICE)
    for i in range(WARMUP_IMAGES):
        with Image.open(rows[i % len(rows)]["image"]) as im: adapter["predict"](im.convert("RGB"))
    sync()
    if DEVICE.startswith("cuda"): torch.cuda.reset_peak_memory_stats(DEVICE)
    for r in rows:
        with Image.open(r["image"]) as im: image = im.convert("RGB")
        # Includes preprocessing, transfer, forward pass, postprocessing and CPU boxes.
        # Excludes disk decoding, model load, tracker, database and video pipeline.
        for rep in range(LATENCY_REPEATS):
            sync(); started=time.perf_counter(); pred=adapter["predict"](image); sync()
            samples.append((time.perf_counter()-started)*1000)
        truth = {"boxes": torch.tensor(r["boxes"], dtype=torch.float32).reshape(-1,4),
                 "labels": torch.tensor(r["labels"], dtype=torch.int64)}
        metric.update([pred], [truth])
        for cid, counts in matching_counts(pred, truth, SCORE_THRESHOLD).items():
            totals[cid] = [a+b for a,b in zip(totals[cid],counts)]
    ap = public_numbers(metric.compute())
    per_class = {}
    for cid, (tp,fp,fn) in totals.items():
        p = tp/(tp+fp) if tp+fp else None
        recall = tp/(tp+fn) if tp+fn else None
        f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None
        per_class[TARGET_CLASSES[cid]] = {"tp":tp,"fp":fp,"fn":fn,"precision":p,"recall":recall,"f1":f1}
    ordered = sorted(samples)
    return {"model":adapter["name"], "split":split, "dataset_sha256":dataset_hash,
            "checkpoint_sha256":checkpoint_hash(adapter["checkpoint"]), "images":len(rows),
            "classes":list(TARGET_CLASSES), "ap":ap, "per_class":per_class,
            "recipe":{"seed":SEED,"epochs":EPOCHS,"yolo_batch":YOLO_BATCH,"rt_batch":RT_BATCH,
                      "rt_grad_accumulation":GRAD_ACCUMULATION,"learning_rate":LEARNING_RATE,
                      "patience":PATIENCE,"ap_max_detections":100,"prediction_max_detections":300},
            "score_threshold":SCORE_THRESHOLD,"inference_score_floor":INFERENCE_SCORE_FLOOR,
            "latency_ms":{"mean":statistics.mean(samples),"p50":statistics.median(samples),
                          "p95":ordered[max(0,math.ceil(.95*len(ordered))-1)],"samples":len(samples)},
            "gpu_peak_allocated_mib":torch.cuda.max_memory_allocated(DEVICE)/2**20 if DEVICE.startswith("cuda") else None,
            "environment":environment(), "promotion_eligible":False}

def train_yolo(name, checkpoint, run_dir):
    from ultralytics import YOLO
    run_dir = Path(run_dir)
    if run_dir.exists(): raise FileExistsError(run_dir)
    model = YOLO(checkpoint)
    model.train(data=str(PREPARED/"dataset.yaml"), project=str(run_dir.parent), name=run_dir.name,
        exist_ok=False, epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=YOLO_BATCH, device=DEVICE,
        workers=0, seed=SEED, deterministic=True, optimizer="AdamW", lr0=LEARNING_RATE,
        patience=PATIENCE, amp=AMP and DEVICE.startswith("cuda"), plots=False, save=True,
        # Keep mosaic off for a predictable first small-data experiment.
        mosaic=0.0, mixup=0.0, close_mosaic=0)
    return str(Path(model.trainer.best).resolve())

def rt_training_batch(processor, rows):
    from PIL import Image
    images, annotations = [], []
    for i,r in enumerate(rows):
        with Image.open(r["image"]) as im: images.append(im.convert("RGB"))
        objects = []
        for b,c in zip(r["boxes"],r["labels"]):
            x1,y1,x2,y2=b
            objects.append({"image_id":i,"category_id":c,"bbox":[x1,y1,x2-x1,y2-y1],
                            "area":(x2-x1)*(y2-y1),"iscrowd":0})
        annotations.append({"image_id":i,"annotations":objects})
    return processor(images=images, annotations=annotations, return_tensors="pt")

def train_rtdetr(checkpoint, records, run_dir, dataset_hash):
    import torch
    from transformers import AutoImageProcessor, AutoModelForObjectDetection
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=False)
    processor = AutoImageProcessor.from_pretrained(checkpoint, local_files_only=True, use_fast=False)
    processor.size={"height":IMAGE_SIZE,"width":IMAGE_SIZE}
    model = AutoModelForObjectDetection.from_pretrained(checkpoint, local_files_only=True,
        id2label=dict(enumerate(TARGET_CLASSES)), label2id={n:i for i,n in enumerate(TARGET_CLASSES)},
        ignore_mismatched_sizes=True).to(DEVICE)
    # Head shapes are rebuilt for the new catalog. This is RT-DETRv2-R18, not RT-DETR-L.
    if model.config.model_type != "rt_detr_v2": raise ValueError("Wrong detector architecture.")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    enabled = AMP and DEVICE.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=enabled)
    train_rows = [r for r in records if r["split"] == "train"]
    if not train_rows: raise ValueError("Empty training split.")
    best_score, stale, history = -math.inf, 0, []
    best = run_dir/"best"
    for epoch in range(EPOCHS):
        model.train(); optimizer.zero_grad(set_to_none=True)
        rows = list(train_rows); random.Random(SEED+epoch).shuffle(rows)
        losses = []
        batches = math.ceil(len(rows)/RT_BATCH)
        for step, start in enumerate(range(0,len(rows),RT_BATCH)):
            batch = rt_training_batch(processor, rows[start:start+RT_BATCH]).to(DEVICE)
            batch["labels"] = [{k:v.to(DEVICE) if hasattr(v,"to") else v for k,v in label.items()}
                               for label in batch["labels"]]
            # Last accumulation window may be shorter than GRAD_ACCUMULATION.
            window_start = (step//GRAD_ACCUMULATION)*GRAD_ACCUMULATION
            divisor = min(GRAD_ACCUMULATION, batches-window_start)
            with torch.autocast(device_type="cuda" if DEVICE.startswith("cuda") else "cpu",
                                dtype=torch.float16 if enabled else torch.bfloat16, enabled=enabled):
                out = model(**batch)
                loss = out.loss / divisor
            if not torch.isfinite(loss): raise RuntimeError("Nonfinite RT-DETR training loss.")
            scaler.scale(loss).backward(); losses.append(float(out.loss.detach().cpu()))
            if (step+1)%GRAD_ACCUMULATION == 0 or step+1 == batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        # Save current epoch, evaluate with the shared validation metric, then keep best.
        current = run_dir/"current"
        model.save_pretrained(current, safe_serialization=True); processor.save_pretrained(current)
        adapter = load_adapter("rtdetr_v2_r18", current, trained=True)
        report = benchmark_adapter(adapter, records, "val", dataset_hash)
        del adapter
        if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
        score = report["ap"]["map"]
        if score is None: raise RuntimeError("Validation mAP undefined; inspect class support.")
        history.append({"epoch":epoch+1,"train_loss":statistics.mean(losses),"val_map":score})
        json_write(run_dir/"history.json", history)
        if score > best_score:
            best_score, stale = score, 0
            model.save_pretrained(best, safe_serialization=True); processor.save_pretrained(best)
        else: stale += 1
        print(f"RT-DETR epoch {epoch+1}: loss={history[-1]['train_loss']:.4f}, val mAP={score:.4f}")
        if stale >= PATIENCE: break
    # current is kept as the last resumable model snapshot; optimizer resume isn't implemented.
    del model
    if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    return str(best.resolve())


In [ ]:
def preview_records(records, count=6):
    from PIL import Image, ImageDraw
    import matplotlib.pyplot as plt
    rows = records[:count]
    fig, axes = plt.subplots(1,len(rows),figsize=(4*len(rows),4), squeeze=False)
    for ax,r in zip(axes[0],rows):
        with Image.open(r["image"]) as im: image=im.convert("RGB")
        draw=ImageDraw.Draw(image)
        for b,c in zip(r["boxes"],r["labels"]):
            draw.rectangle(b,outline="red",width=3); draw.text((b[0],b[1]),TARGET_CLASSES[c],fill="red")
        ax.imshow(image); ax.set_title(r["split"]); ax.axis("off")
    plt.show()

def results_table(results):
    import pandas as pd
    return pd.DataFrame([{"model":name, "mAP50-95":r["ap"]["map"],"AP50":r["ap"]["map_50"],
                          "latency_p95_ms":r["latency_ms"]["p95"],"images":r["images"],
                          "checkpoint_sha256":r["checkpoint_sha256"]}
                         for name,r in results.items()]).sort_values(
                             ["mAP50-95","latency_p95_ms"], ascending=[False,True]) if results else pd.DataFrame()

def export_candidate(name, checkpoint, validation, test, dataset_hash, destination):
    if test.get("split") != "test" or validation.get("split") != "val":
        raise ValueError("Candidate export requires independent validation and final test reports.")
    digest = checkpoint_hash(checkpoint)
    if validation.get("score_threshold") != test.get("score_threshold"):
        raise ValueError("Evaluation score thresholds differ.")
    for report in [validation,test]:
        if report.get("model") != name or report.get("classes") != TARGET_CLASSES:
            raise ValueError("Report model/class mismatch.")
        if report.get("dataset_sha256") != dataset_hash or report.get("checkpoint_sha256") != digest:
            raise ValueError("Dataset/checkpoint changed after evaluation.")
        if any(report["per_class"][c]["tp"]+report["per_class"][c]["fn"] == 0 for c in TARGET_CLASSES):
            raise ValueError("Every target must have validation/test ground truth before export.")
    destination=Path(destination)
    if destination.exists(): raise FileExistsError(destination)
    destination.parent.mkdir(parents=True,exist_ok=True)
    metadata={"version":f"{name}-{RUN_ID}","dataset_version":dataset_hash,"sha256":digest,
              "classes":TARGET_CLASSES,"adapter_required":name=="rtdetr_v2_r18",
              "promotion_eligible":False,"status":"research_candidate"}
    with zipfile.ZipFile(destination,"x",compression=zipfile.ZIP_DEFLATED) as z:
        path=Path(checkpoint)
        if path.is_file(): z.write(path,"model.pt")
        else:
            for p in sorted(path.rglob("*")):
                if p.is_file(): z.write(p,"hf_model/"+p.relative_to(path).as_posix())
        z.writestr("metadata.json",json.dumps(metadata,indent=2))
        z.writestr("validation.json",json.dumps(validation,indent=2,allow_nan=False))
        z.writestr("test.json",json.dumps(test,indent=2,allow_nan=False))
        z.writestr("README.txt",
            "Research candidate, not promoted. No training images included.\n"
            "YOLO: inspect model.names and match application class configuration before integration.\n"
            "RT-DETR: requires a separately tested application adapter; do not rename to .pt.\n"
            "Detector metrics alone do not validate tracking, occupancy, preparation or sales.\n")
    return destination

def video_preview(adapter, source, destination, max_frames=300):
    import cv2
    from PIL import Image
    source, destination=Path(source).resolve(),Path(destination).resolve()
    if source==destination or destination.exists(): raise ValueError("Refusing to overwrite video.")
    cap=cv2.VideoCapture(str(source)); writer=None; count=0
    if not cap.isOpened(): raise ValueError("Cannot decode source video.")
    try:
        fps=cap.get(cv2.CAP_PROP_FPS)
        if not math.isfinite(fps) or fps<=0: raise ValueError("Unknown source FPS.")
        while count<max_frames:
            ok,frame=cap.read()
            if not ok: break
            if writer is None:
                destination.parent.mkdir(parents=True,exist_ok=True)
                writer=cv2.VideoWriter(str(destination),cv2.VideoWriter_fourcc(*"mp4v"),fps,
                                       (frame.shape[1],frame.shape[0]))
                if not writer.isOpened(): raise RuntimeError("Video encoder unavailable.")
            pred=adapter["predict"](Image.fromarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)))
            for b,s,c in zip(pred["boxes"].tolist(),pred["scores"].tolist(),pred["labels"].tolist()):
                if s<SCORE_THRESHOLD: continue
                x1,y1,x2,y2=map(round,b)
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.putText(frame,f"{TARGET_CLASSES[c]} {s:.2f}",(x1,max(15,y1-4)),
                            cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,0),1)
            writer.write(frame); count+=1
        if count==0: raise ValueError("Video contains no decodable frames.")
    finally:
        cap.release()
        if writer is not None: writer.release()
    return {"frames":count,"output":str(destination),"meaning":"detections only; no event/sales claims"}


In [ ]:
if RUN_DOWNLOADS or DATASET_ROOT is not None:
    if REAL_CAFE_DATA and GROUPS_CSV is None:
        raise ValueError("Own video frames require GROUPS_CSV with recording/session IDs.")
    groups=load_groups(GROUPS_CSV)
    if SOURCE=="hf_coco":
        if not RUN_DOWNLOADS: raise ValueError("HF acquisition needs RUN_DOWNLOADS=True.")
        RECORDS=acquire_hf(HF_TRAIN_LIMIT,HF_TEST_LIMIT,RUN_DIR/"source-hf")
    elif SOURCE=="kaggle_cup":
        if TARGET_CLASSES != ["cup"]:
            raise ValueError("Cup dataset is cup-only: set TARGET_CLASSES=['cup'].")
        root=DATASET_ROOT
        if root is None:
            import kagglehub
            root=kagglehub.dataset_download(KAGGLE_CUP)
        # Inspect sample TXT IDs first. YOLO_CLASS_NAMES is an explicit source mapping.
        RECORDS=import_yolo(root,YOLO_CLASS_NAMES,YOLO_FULLY_ANNOTATED,groups,
                            ALLOW_EMPTY_LABELS,LOCAL_LIMIT_PER_SPLIT)
    elif SOURCE=="yolo":
        if DATASET_ROOT is None: raise ValueError("Set DATASET_ROOT.")
        RECORDS=import_yolo(DATASET_ROOT,YOLO_CLASS_NAMES,YOLO_FULLY_ANNOTATED,groups,
                            ALLOW_EMPTY_LABELS,LOCAL_LIMIT_PER_SPLIT)
    elif SOURCE=="coco_json":
        if DATASET_ROOT is None: raise ValueError("Attach Kaggle COCO and set DATASET_ROOT.")
        RECORDS=import_coco_json(DATASET_ROOT,LOCAL_LIMIT_PER_SPLIT,groups)
    else: raise ValueError("Unknown source.")
    RECORDS,AUDIT=split_and_audit(RECORDS,SEED)
    RECORDS,DATASET_HASH=prepare_dataset(RECORDS,PREPARED)
    json_write(RUN_DIR/"dataset_audit.json",AUDIT)
    print(json.dumps(AUDIT,indent=2))
    preview_records(RECORDS)
else:
    print("Data acquisition disabled. Set DATASET_ROOT or explicitly enable RUN_DOWNLOADS.")

## Pretrained baseline, then training
Use baseline results only on **validation** while developing. COCO-pretrained models have already seen COCO training photos, and authors often use COCO validation to choose checkpoints: this is transfer-learning practice data, not independent café evidence.

Training uses 20 epochs as a starting budget, not a promise of convergence. YOLO uses its native best checkpoint criterion; RT-DETR uses shared validation mAP. Recipes and preprocessing differ and are recorded; use multiple seeds and longer runs before concluding that an architecture is superior. CUDA OOM: start a new run at batch 1 / 384 px or disable AMP if numerically unstable. Never silently change settings midway through a comparison.

VLMs remain outside this supervised detector leaderboard: they need a separately labeled temporal/question-answer benchmark. The earlier SmolVLM probe was prompt-sensitive and too slow for every-frame detection on this laptop. A later hybrid should call a VLM only on selected ambiguous events, with abstention and measured cost/accuracy.

In [ ]:
if RUN_PRETRAINED:
    if not RECORDS: raise ValueError("Prepare the dataset first.")
    import torch
    PRETRAINED_RESULTS={}
    for name in MODEL_NAMES:
        adapter=load_adapter(name,pretrained_location(name))
        PRETRAINED_RESULTS[name]=benchmark_adapter(adapter,RECORDS,"val",DATASET_HASH)
        json_write(RUN_DIR/f"baseline-{name}.json",PRETRAINED_RESULTS[name])
        del adapter
        if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    display(results_table(PRETRAINED_RESULTS))

In [ ]:
if RUN_TRAINING:
    if not RECORDS: raise ValueError("Prepare the dataset first.")
    import torch
    torch.manual_seed(SEED); random.seed(SEED)
    assert_dataset_unchanged(RECORDS)
    for name in MODEL_NAMES:
        checkpoint=pretrained_location(name)
        output=RUN_DIR/"training"/name
        trained=(train_rtdetr(checkpoint,RECORDS,output,DATASET_HASH) if name=="rtdetr_v2_r18"
                 else train_yolo(name,checkpoint,output))
        ARTIFACTS[name]=trained
        json_write(RUN_DIR/"artifacts.json",ARTIFACTS)
        adapter=load_adapter(name,trained,trained=True)
        VALIDATION_RESULTS[name]=benchmark_adapter(adapter,RECORDS,"val",DATASET_HASH)
        json_write(RUN_DIR/f"validation-{name}.json",VALIDATION_RESULTS[name])
        del adapter
        if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    display(results_table(VALIDATION_RESULTS))

## Freeze candidates and evaluate test once
Enable `RUN_FINAL_TEST` only after the training recipes, target classes and score threshold are fixed. This saves a hash-bound freeze record and refuses a second test run in the same directory. A new run does not make a previously inspected test set independent again: use a new held-out café recording for the next final assessment.

Compare per-class AP, recall and false positives along with p95 latency; a high average can hide a weak cup detector. Absent ground-truth classes produce undefined metrics and block candidate export. Small samples need more data, not more decimal places.

In [ ]:
if RUN_FINAL_TEST:
    if not ARTIFACTS or set(ARTIFACTS)!=set(VALIDATION_RESULTS):
        raise ValueError("Train and validate the candidates first.")
    freeze=RUN_DIR/"frozen_candidates.json"
    if freeze.exists(): raise FileExistsError("Final test already started for this run.")
    frozen={"dataset_sha256":DATASET_HASH,"score_threshold":SCORE_THRESHOLD,
            "image_size":IMAGE_SIZE,"amp":AMP,"classes":TARGET_CLASSES,
            "checkpoints":{n:checkpoint_hash(p) for n,p in ARTIFACTS.items()},
            "selection_split":"val","selection":results_table(VALIDATION_RESULTS).to_dict("records")}
    json_write(freeze,frozen)
    import torch
    for name,path in ARTIFACTS.items():
        if checkpoint_hash(path)!=frozen["checkpoints"][name]: raise ValueError("Frozen weights changed.")
        adapter=load_adapter(name,path,trained=True)
        TEST_RESULTS[name]=benchmark_adapter(adapter,RECORDS,"test",DATASET_HASH)
        json_write(RUN_DIR/f"test-{name}.json",TEST_RESULTS[name])
        del adapter
        if DEVICE.startswith("cuda"): torch.cuda.empty_cache()
    display(results_table(TEST_RESULTS))

## Export research candidates for later integration
Exports contain weights, metadata and evaluation reports, **not source footage**. They are not automatically copied into `models/`, committed, or promoted. YOLO weights need matching application class configuration and runtime compatibility tests; YOLO26 also needs the newer Ultralytics runtime. RT-DETR exports a Hugging Face directory and requires a new operational adapter.

Keep the best candidate based on the frozen validation decision, report final test results honestly, then label and evaluate real overhead café sessions. The application's promotion gate additionally requires independent held-out tracking/occupancy/event evidence; these detector results alone cannot satisfy it.

In [ ]:
if RUN_EXPORT:
    if not TEST_RESULTS: raise ValueError("Run the frozen final test first.")
    for name,path in ARTIFACTS.items():
        bundle=export_candidate(name,path,VALIDATION_RESULTS[name],TEST_RESULTS[name],
                                DATASET_HASH,RUN_DIR/"exports"/f"{name}-candidate.zip")
        print(bundle)

if RUN_VIDEO:
    if VIDEO_PATH is None or VIDEO_MODEL not in ARTIFACTS:
        raise ValueError("Set VIDEO_PATH and train/select VIDEO_MODEL first.")
    adapter=load_adapter(VIDEO_MODEL,ARTIFACTS[VIDEO_MODEL],trained=True)
    print(video_preview(adapter,VIDEO_PATH,RUN_DIR/f"{VIDEO_MODEL}-preview.mp4",VIDEO_MAX_FRAMES))
    del adapter